# Convolutional Neural Networks

**Goal:** implement a 2-D convolution forward pass from scratch, validate it against PyTorch, visualise edge-detection kernels, implement max-pooling from scratch, train a tiny CNN on synthetic images (no downloads), and explain parameter sharing / translation equivariance.

## Configuration

Device, seed, and dtype come from the repo's `config.toml` via `shared.config.configure()` — never hardcoded.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # noqa
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402
import torch.nn as nn  # noqa: E402
import torch.nn.functional as F  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## 2-D Convolution — from scratch

The output of a 2-D convolution is:

```
Y[b, co, i, j] = sum_{ci} sum_u sum_v  X[b, ci, i*stride + u - pad, j*stride + v - pad]
                                         * K[co, ci, u, v]
               + bias[co]
```

Output spatial size:

```
Hout = floor((H + 2*pad_h - Kh) / stride_h + 1)
Wout = floor((W + 2*pad_w - Kw) / stride_w + 1)
```

We implement this using `torch.Tensor.unfold` (vectorised, still from-scratch — no autograd, just raw tensor ops).

In [2]:
import math


def conv2d_scratch(
    x: torch.Tensor,
    weight: torch.Tensor,
    bias: torch.Tensor | None = None,
    stride: int = 1,
    padding: int = 0,
) -> torch.Tensor:
    """2-D convolution forward pass implemented via unfold (im2col-style).

    Extracts every (Kh × Kw) patch at each stride position into a column matrix,
    then performs a single matrix multiply against the flattened kernels.

    Args:
        x: Input tensor of shape (B, Cin, H, W).
        weight: Kernel tensor of shape (Cout, Cin, Kh, Kw).
        bias: Optional bias of shape (Cout,).
        stride: Stride applied to both spatial dimensions.
        padding: Zero-padding added to both spatial dimensions.

    Returns:
        Output tensor of shape (B, Cout, Hout, Wout).
    """
    B, Cin, H, W = x.shape
    Cout, _Cin, Kh, Kw = weight.shape
    assert _Cin == Cin, f"channel mismatch: x has {Cin}, weight expects {_Cin}"

    Hout = math.floor((H + 2 * padding - Kh) / stride + 1)
    Wout = math.floor((W + 2 * padding - Kw) / stride + 1)

    # Zero-pad the input along spatial dimensions
    if padding > 0:
        x_padded = F.pad(x, (padding, padding, padding, padding), value=0.0)
    else:
        x_padded = x

    # unfold extracts sliding patches: (B, Cin, Hout, Wout, Kh, Kw)
    x_unf = x_padded.unfold(2, Kh, stride).unfold(3, Kw, stride)

    # Rearrange to (B, Hout*Wout, Cin*Kh*Kw): each row = one spatial position's patch
    x_col = x_unf.permute(0, 2, 3, 1, 4, 5).contiguous().view(B, Hout * Wout, Cin * Kh * Kw)

    # Flatten kernel to (Cout, Cin*Kh*Kw), transpose for matmul
    w_flat = weight.view(Cout, Cin * Kh * Kw)

    # (B, HW, CKK) @ (CKK, Cout) → (B, HW, Cout) → (B, Cout, Hout, Wout)
    out = (x_col @ w_flat.t()).permute(0, 2, 1).contiguous().view(B, Cout, Hout, Wout)

    if bias is not None:
        out = out + bias.view(1, Cout, 1, 1)
    return out


# ── Quick smoke test ─────────────────────────────────────────────────────────
torch.manual_seed(42)
x_test = torch.randn(2, 3, 8, 8, device=device)
w_test = torch.randn(8, 3, 3, 3, device=device)
b_test = torch.randn(8, device=device)

out_scratch = conv2d_scratch(x_test, w_test, b_test, stride=1, padding=1)
out_ref     = F.conv2d(x_test, w_test, b_test, stride=1, padding=1)

print("scratch  :", out_scratch.shape)
print("F.conv2d :", out_ref.shape)
print("max abs error:", (out_scratch - out_ref).abs().max().item())

scratch  : torch.Size([2, 8, 8, 8])
F.conv2d : torch.Size([2, 8, 8, 8])
max abs error: 2.86102294921875e-06


## Validation — from-scratch vs `F.conv2d`

Assert on multiple shapes, strides, and paddings.

In [3]:
def _check(B, Cin, H, W, Cout, Kh, Kw, stride, padding, seed=0):
    torch.manual_seed(seed)
    x = torch.randn(B, Cin, H, W, device=device)
    w = torch.randn(Cout, Cin, Kh, Kw, device=device)
    b = torch.randn(Cout, device=device)

    got  = conv2d_scratch(x, w, b, stride=stride, padding=padding)
    want = F.conv2d(x, w, b, stride=stride, padding=padding)

    assert got.shape == want.shape, f"shape mismatch: {got.shape} vs {want.shape}"
    assert torch.allclose(got, want, atol=1e-4), (
        f"values differ  max_err={( got - want).abs().max().item():.2e}"
    )
    return True


configs = [
    # B, Cin, H, W, Cout, Kh, Kw, stride, padding
    (1, 1,  8,  8, 1, 3, 3, 1, 0),   # minimal
    (2, 3, 16, 16, 8, 3, 3, 1, 1),   # multi-channel, same-padding
    (4, 3, 12, 12, 16, 5, 5, 2, 2),  # stride 2
    (1, 1, 10, 10, 4, 3, 3, 2, 0),   # stride 2, no padding
    (2, 2, 14, 14, 6, 1, 1, 1, 0),   # 1×1 kernel
]

for cfg in configs:
    _check(*cfg)
    print("PASS", cfg)

print("\nAll conv2d_scratch validations passed.")

PASS (1, 1, 8, 8, 1, 3, 3, 1, 0)
PASS (2, 3, 16, 16, 8, 3, 3, 1, 1)


PASS (4, 3, 12, 12, 16, 5, 5, 2, 2)


PASS (1, 1, 10, 10, 4, 3, 3, 2, 0)
PASS (2, 2, 14, 14, 6, 1, 1, 1, 0)

All conv2d_scratch validations passed.


## Idiomatic PyTorch: `nn.Conv2d`

`nn.Conv2d` wraps all the same logic with autograd support, bias initialisation, and
optional groups/dilation.

In [4]:
torch.manual_seed(0)
conv = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, stride=1, padding=1).to(device)

x_demo = torch.randn(1, 3, 8, 8, device=device)
y_demo = conv(x_demo)
print(f"input : {x_demo.shape}  →  output : {y_demo.shape}")
print(f"weight: {conv.weight.shape}, bias: {conv.bias.shape}")
params_conv = sum(p.numel() for p in conv.parameters())
params_fc   = 3 * 8 * 8 * (8 * 8 * 8)      # flattened in → out
print(f"\nConv2d parameters  : {params_conv}")
print(f"Equivalent Linear  : {params_fc}  ({params_fc // params_conv}× more)")

input : torch.Size([1, 3, 8, 8])  →  output : torch.Size([1, 8, 8, 8])
weight: torch.Size([8, 3, 3, 3]), bias: torch.Size([8])

Conv2d parameters  : 224
Equivalent Linear  : 98304  (438× more)


## Receptive Field Growth with Stacked Convolutions

Stacking small filters grows the receptive field so deep features see large input regions while keeping few parameters.

For stacked `k×k` convolutions with stride 1 and no dilation:
```
RF = 1 + n * (k - 1)
```

In [5]:
# Stacked k×k convs (stride 1, no dilation): RF = 1 + n*(k-1)
for n in [1, 2, 3, 4]:
    print(f"{n} conv(3x3) layers -> receptive field {1 + n*(3-1)}x{1 + n*(3-1)}")

1 conv(3x3) layers -> receptive field 3x3
2 conv(3x3) layers -> receptive field 5x5
3 conv(3x3) layers -> receptive field 7x7
4 conv(3x3) layers -> receptive field 9x9


## Conv concepts — edge detection on a synthetic image

A Sobel-X kernel detects vertical edges. We apply it to a hand-crafted 1-channel image and
visualise the input vs the resulting feature map.

In [6]:
# Synthetic 1-channel image: a white rectangle on black background
torch.manual_seed(1)
img = torch.zeros(1, 1, 16, 16, device=device)
img[0, 0, 4:12, 6:10] = 1.0          # white rectangle

# Sobel-X kernel (detects vertical edges)
sobel_x = torch.tensor(
    [[-1., 0., 1.],
     [-2., 0., 2.],
     [-1., 0., 1.]],
    device=device
).view(1, 1, 3, 3)

# Apply via F.conv2d
feature_map = F.conv2d(img, sobel_x, padding=1)

# Visualise (on CPU for matplotlib)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(img[0, 0].cpu(), cmap="gray")
axes[0].set_title("Input image")
axes[0].axis("off")
axes[1].imshow(feature_map[0, 0].cpu(), cmap="RdBu")
axes[1].set_title("Sobel-X feature map")
axes[1].axis("off")
plt.tight_layout()
plt.savefig("conv_visualization.png", dpi=100)
plt.close()
print("Saved conv_visualization.png")
print("feature map shape:", feature_map.shape)

Saved conv_visualization.png
feature map shape: torch.Size([1, 1, 16, 16])


## Max Pooling — from scratch

Max pooling slides a window of size `(kH, kW)` over the spatial dimensions and takes the
maximum value in each window, reducing the spatial resolution by a factor equal to the stride.

In [7]:
def max_pool2d_scratch(
    x: torch.Tensor,
    kernel_size: int = 2,
    stride: int | None = None,
) -> torch.Tensor:
    """Max pooling via unfold.

    Args:
        x: Input of shape (B, C, H, W).
        kernel_size: Square pool window size.
        stride: Defaults to kernel_size (non-overlapping).

    Returns:
        Pooled tensor of shape (B, C, Hout, Wout).
    """
    if stride is None:
        stride = kernel_size

    B, C, H, W = x.shape
    k = kernel_size
    Hout = (H - k) // stride + 1
    Wout = (W - k) // stride + 1

    # unfold to extract patches: (B, C, Hout, Wout, k, k)
    patches = x.unfold(2, k, stride).unfold(3, k, stride)
    # flatten the patch dim and take max
    out = patches.contiguous().view(B, C, Hout, Wout, k * k).max(dim=-1).values
    return out


# Validation
torch.manual_seed(7)
x_pool = torch.randn(3, 4, 12, 12, device=device)

for ks, st in [(2, 2), (3, 3), (2, 1), (3, 2)]:
    got  = max_pool2d_scratch(x_pool, kernel_size=ks, stride=st)
    want = F.max_pool2d(x_pool, kernel_size=ks, stride=st)
    assert got.shape == want.shape, f"shape {got.shape} vs {want.shape}"
    assert torch.allclose(got, want, atol=1e-4), (
        f"max_pool mismatch ks={ks} st={st}  max_err={( got - want).abs().max().item():.2e}"
    )
    print(f"PASS  max_pool2d  ks={ks}  stride={st}  →  {tuple(got.shape)}")

print("\nAll max_pool2d_scratch validations passed.")

PASS  max_pool2d  ks=2  stride=2  →  (3, 4, 6, 6)
PASS  max_pool2d  ks=3  stride=3  →  (3, 4, 4, 4)
PASS  max_pool2d  ks=2  stride=1  →  (3, 4, 11, 11)
PASS  max_pool2d  ks=3  stride=2  →  (3, 4, 5, 5)

All max_pool2d_scratch validations passed.


## Tiny CNN — trained on synthetic 1×12×12 images

We generate a small binary-classification dataset entirely in-notebook (no downloads):

- **Class 0**: images with a horizontal bar
- **Class 1**: images with a vertical bar

The CNN has two conv layers and a single linear head.

In [8]:
# ── Dataset generation ─────────────────────────────────────────────────────
def make_dataset(n_per_class: int = 150, img_size: int = 12, seed: int = 0) -> tuple:
    """Return (X, y) where X is (N, 1, H, W) float and y is (N,) long.

    Class 0 = random horizontal bar; class 1 = random vertical bar.
    Gaussian noise added so the task is non-trivial.
    """
    rng = torch.Generator()
    rng.manual_seed(seed)
    imgs, labels = [], []
    for cls in range(2):
        for _ in range(n_per_class):
            img = torch.zeros(1, img_size, img_size)
            if cls == 0:                          # horizontal bar
                row = torch.randint(1, img_size - 1, (1,), generator=rng).item()
                img[0, row, :] = 1.0
            else:                                 # vertical bar
                col = torch.randint(1, img_size - 1, (1,), generator=rng).item()
                img[0, :, col] = 1.0
            img += 0.15 * torch.randn(1, img_size, img_size, generator=rng)
            imgs.append(img)
            labels.append(cls)
    X = torch.stack(imgs)
    y = torch.tensor(labels, dtype=torch.long)
    # shuffle
    perm = torch.randperm(len(y), generator=rng)
    return X[perm].to(device), y[perm].to(device)


X_all, y_all = make_dataset(n_per_class=150, img_size=12, seed=0)
split = int(0.8 * len(X_all))
X_train, y_train = X_all[:split], y_all[:split]
X_val,   y_val   = X_all[split:], y_all[split:]
print(f"train: {X_train.shape}  val: {X_val.shape}  classes: {y_all.unique().tolist()}")

train: torch.Size([240, 1, 12, 12])  val: torch.Size([60, 1, 12, 12])  classes: [0, 1]


In [9]:
# ── Model definition ────────────────────────────────────────────────────────
class TinyCNN(nn.Module):
    """Small CNN: two conv layers + a linear classifier."""

    def __init__(self) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8,  kernel_size=3, padding=1)   # 1×12×12 → 8×12×12
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)   # 8×6×6  → 16×6×6
        self.pool  = nn.MaxPool2d(2, 2)                            # halves spatial dims
        self.fc    = nn.Linear(16 * 3 * 3, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(F.relu(self.conv1(x)))   # 1×12×12 → 8×6×6
        x = self.pool(F.relu(self.conv2(x)))   # 8×6×6  → 16×3×3
        x = x.flatten(1)                       # 144
        return self.fc(x)


model = TinyCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"TinyCNN  total parameters: {total_params}")

# Dense baseline for comparison (no spatial structure)
dense_params = (1 * 12 * 12) * 64 + 64 + 64 * 2 + 2
print(f"Baseline Linear (144→64→2): {dense_params} params — CNN is {dense_params // total_params}× smaller")

TinyCNN  total parameters: 1538
Baseline Linear (144→64→2): 9410 params — CNN is 6× smaller


In [10]:
# ── Training loop ────────────────────────────────────────────────────────────
BATCH = 64
EPOCHS = 20
LR = 3e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

history_loss  = []
history_acc   = []

torch.manual_seed(42)
for epoch in range(1, EPOCHS + 1):
    model.train()
    # Mini-batch SGD over training set
    perm = torch.randperm(len(X_train), device=device)
    epoch_loss = 0.0
    for start in range(0, len(X_train), BATCH):
        idx  = perm[start: start + BATCH]
        xb, yb = X_train[idx], y_train[idx]
        logits = model(xb)
        loss   = criterion(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(yb)

    avg_loss = epoch_loss / len(X_train)

    # Train accuracy
    model.eval()
    with torch.no_grad():
        preds = model(X_train).argmax(1)
        acc   = (preds == y_train).float().mean().item()

    history_loss.append(avg_loss)
    history_acc.append(acc)

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:3d}  loss={avg_loss:.4f}  train_acc={acc:.3f}")

initial_loss = history_loss[0]
final_loss   = history_loss[-1]
final_acc    = history_acc[-1]
assert final_loss < initial_loss * 0.5, (
    f"loss did not decrease enough: {initial_loss:.4f} → {final_loss:.4f} (need < 50% of initial)"
)
print(f"\nFinal train accuracy: {final_acc:.3f}  (initial loss {initial_loss:.4f} → final {final_loss:.4f})")

epoch   1  loss=0.6851  train_acc=0.967
epoch   5  loss=0.3980  train_acc=1.000
epoch  10  loss=0.0188  train_acc=1.000
epoch  15  loss=0.0015  train_acc=1.000
epoch  20  loss=0.0007  train_acc=1.000

Final train accuracy: 1.000  (initial loss 0.6851 → final 0.0007)


In [11]:
# ── Plot training curves ────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3))

ax1.plot(range(1, EPOCHS + 1), history_loss, marker="o", markersize=3)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-entropy loss")
ax1.set_title("Training loss")

ax2.plot(range(1, EPOCHS + 1), [a * 100 for a in history_acc], marker="o", markersize=3, color="tab:green")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Train accuracy")
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.savefig("tiny_cnn_training.png", dpi=100)
plt.close()
print("Saved tiny_cnn_training.png")

Saved tiny_cnn_training.png


## The idiomatic PyTorch way

`nn.Conv2d`, `nn.MaxPool2d`, and `nn.Linear` compose cleanly inside `nn.Sequential` or a
custom `nn.Module`. The `TinyCNN` above *is* the idiomatic approach. The from-scratch
`conv2d_scratch` function was only for pedagogical purposes — use `F.conv2d` or `nn.Conv2d`
in production.

In [12]:
# Evaluate on the held-out validation split
model.eval()
with torch.no_grad():
    val_logits = model(X_val)
    val_preds  = val_logits.argmax(1)
    val_acc    = (val_preds == y_val).float().mean().item()

print(f"Validation accuracy: {val_acc:.3f}")
assert val_acc > 0.75, f"Expected val_acc > 0.75, got {val_acc:.3f}"

Validation accuracy: 1.000


## Parameter sharing and translation equivariance

**Weight sharing** means the same kernel is applied at every spatial location. A `3×3` filter
on a 3-channel image uses only `3×3×3 = 27` weights regardless of the image resolution.
A fully-connected layer from a `3×32×32` image to 64 outputs needs `3×32×32×64 = 196,608`
weights — roughly 7,000× more.

**Translation equivariance** follows directly: if the input shifts by `(Δh, Δw)`, the feature
map shifts by `(Δh/stride, Δw/stride)` (same outputs, just relocated). Formally:

```
conv(shift(X, Δ)) == shift(conv(X), Δ/stride)
```

This means a bar-detector learned at position (3, 5) works equally well at (7, 9) — the model
does not need to re-learn the same concept at every position.

**Why CNNs beat dense nets on images:**

1. *Parameter efficiency* — weight sharing keeps the model small even for large inputs.
2. *Inductive bias* — locality and equivariance match the structure of natural images.
3. *Generalization* — fewer parameters with the right bias generalizes better from limited data.

See also: `[[backpropagation]]` for how gradients flow through conv layers during training.

## Takeaways

- A 2-D convolution is a dot-product between a sliding kernel and local patches; output size is
  `floor((H + 2p - k) / s + 1)`.
- **`unfold`** (im2col) turns convolution into a matrix multiply, making the scratch
  implementation numerically identical to `F.conv2d` (atol 1e-4).
- Max pooling selects the maximum activation in each window, discarding precise location while
  preserving the presence of a feature.
- **Weight sharing** is why a `3×3` conv on any resolution costs the same 9 weights per
  input channel per output channel.
- **Translation equivariance**: shift the input → shift the feature map. CNNs exploit this for
  free; dense layers must learn each position separately.
- A tiny two-layer CNN converges to high accuracy on a synthetic bar-classification task in
  a few epochs, while using far fewer parameters than an equivalent dense network.